# **PLEASE JOIN THE COMMUNITY COMP, NO EXTRA WORK IS REQUIRED:**
**https://www.kaggle.com/competitions/march-machine-learning-mania-2026-semi-spherical-scoring**

# **IMPORTANT NOTICE:**

After Selection Sunday, I will be running the full algorithm, [which directly produced 70 silver medals on Kaggle last year](https://www.kaggle.com/code/kaito510/updated-goto-conversion-winning-solution), to predict both the Men’s and Women’s March Madness tournaments. But because it has significantly more computations than the approximation algorithm I have been using for the weekly articles, expect the next iteration of articles to be posted slightly longer than a week after the last iteration.

Please subscribe to my **free** Substack (linked below↓) to get notified as soon as it is published

**https://gotoconversion.substack.com/**

# Abstract

As shown on my [**Kaggle profile**](https://www.kaggle.com/kaito510), this solution was the key ingredient for **my medals every year from 2019 to 2022 and 2024 to 2025; with a gold medal in 2022**. In 2023, my model performance was still at medal-level; see the [What-If Leaderboard section](https://www.kaggle.com/code/kaito510/goto-conversion-winning-solution#What-If-Leaderboard-for-2023) for details.

The only model used in this solution is [**goto_conversion**](https://github.com/gotoConversion/goto_conversion), and the only data it uses is tournament progression probabilities. From 2019 to 2023, 538's tournament progression probabilities were used; in 2024 and 2025, this solution used tournament progression probabilities computed by converting betting odds to outcome probabilities using [goto_conversion](https://github.com/gotoConversion/goto_conversion), which can be freely found [here](https://github.com/gotoConversion/goto_conversion/tree/main/goto_conversion/probabilityTables) as csv files.

**In 2024, this solution was publicised for the first time, and it was sufficient for a bronze medal with direct submission**. This can be verified by noticing [2024's leaderboard scores from 86th to 100th](https://www.kaggle.com/competitions/march-machine-learning-mania-2024/leaderboard) and the leaderboard score of [2024's version of this solution](https://www.kaggle.com/code/kaito510/updated-1xgold-2xsilvers-key-ingredient) are both 0.06035.

**In 2025, this solution alone was sufficient for a silver medal with direct submission**. This can be verified by noticing [2025's leaderboard scores from 14th to 83rd](https://www.kaggle.com/competitions/march-machine-learning-mania-2025/leaderboard) and the leaderboard score of [2025's version of this solution](https://www.kaggle.com/code/kaito510/updated-goto-conversion-winning-solution) are both 0.10921.

Overall, this solution has been used to produce a total of $47,000 in prize money, over 10 gold medals and over 100 medals on Kaggle listed [here](https://github.com/gotoConversion/goto_conversion?tab=readme-ov-file#wins).

goto_conversion is originally designed for converting betting odds to outcome probabilities. It easily outperforms the multiplicative method (normalisation of inverse odds) because the mulitplicative method does not consider the favourite-longshot bias. The favourite-longshot bias states that the betting odds for more likely events have lower expected loss than less likely events; vice-versa. Other existing methods such as Shin and Power methods have attempted to consider the favourite-longshot bias but the goto_conversion outperforms them.

Now, why is the favourite-longshot bias relevant to this competition? Because team ratings use regular season data to measure strength of teams at March Madness. This underrates the stronger teams because they have more “garbage” minutes and even entire games during the regular season where they rest their best players without the risk of losing. In other words, with opponent quality adjusted, stronger teams tend to have less dominant statistics during the regular season compared to during March Madness. Because during March Madness, almost every game is reasonably competitive and thus there is significantly less "garbage" compared to during the regular season. Notice underrated stronger teams is a problem similar to the favourite-longshot bias because if we use the multiplicative method to convert tournament progression probabilities (derived by ratings) to match probabilities (required for this competition), we will underestimate the probability of stronger teams winning; just like how if we use the multiplicative method to convert betting odds to probabilities, we will underestimate the probability of stronger teams winning (favourite-longshot bias).

For example, to compute the probability of Team A beating Team B and given they will play each other in the second round if it was to happen, we will compute the probability for this match by:

*P(Team A wins 2nd round match) = P(Team A reaching 3rd round) / P(Team A reaching 2nd round)*

*P(Team B wins 2nd round match) = P(Team B reaching 3rd round) / P(Team B reaching 2nd round)*

Notice *P(Team A wins 2nd round match)* and *P(Team B wins 2nd round match)* will not necessarily sum to 1. **This is the problem!** As discussed, we should convert these tournament progression probabilities into match probabilities using the goto_conversion to ensure they sum to 1 while considering the favourite-longshot bias.

*goto_conversion([1/P(Team A wins 2nd round match), 1/P(Team B wins 2nd round match)])*

*= [P(Team A beats Team B), P(Team B beats Team A)]*

Recall, if we use the multiplicative conversion instead of the goto_conversion, we will underestimate the probability of the stronger team winning because multiplicative conversion does not consider the favourite-longshot bias.

As stated [here](https://fivethirtyeight.com/features/how-our-march-madness-predictions-work-2/), 538 have also noticed that their ratings imply fewer upsets during March Madness compared to the regular season. But they believe the main reason is because March Madness is played under better and fairer conditions.

In [1]:
#Get 538-style probability matrices extracted from goto_conversion repo:
#https://github.com/gotoConversion/goto_conversion
#Matrices were computed by converting betting odds to probabilities using goto_conversion

import pandas as pd
import numpy as np
import plotly.express as px
    
kaggleFolderPath = '/kaggle/input/competitions/march-machine-learning-mania-2026'
fivethirtyeightFolderPath = '/kaggle/input/538data'

In [2]:
#Mens Probability Matrix

mensProbabilities_df = pd.read_csv(fivethirtyeightFolderPath + '/mensProbabilitiesTable.csv', index_col = 'player') #source: https://github.com/gotoConversion/goto_conversion
mensProbabilities_df = mensProbabilities_df.drop('Elo_Rating', axis=1)

In [3]:
#Womens Probability Matrix

womensProbabilities_df = pd.read_csv(fivethirtyeightFolderPath + '/womensProbabilitiesTable.csv', index_col = 'player') #source: https://github.com/gotoConversion/goto_conversion
womensProbabilities_df = womensProbabilities_df.drop('Elo_Rating', axis=1)

# Experiment

In [4]:
#Install libraries

%pip install goto-conversion

Note: you may need to restart the kernel to use updated packages.


In [5]:
#Load libraries

import goto_conversion
import math
import os
import statistics
import copy

In [6]:
#Import each year's 538 files before it went behind a paywall

both2019_df = pd.read_csv(fivethirtyeightFolderPath + '/fivethirtyeight_ncaa_forecasts (2).csv') #538 file for 2019
both2021_df = pd.read_csv(fivethirtyeightFolderPath + '/fivethirtyeight_ncaa_forecasts (3).csv') #538 file for 2021
both2022_df = pd.read_csv(fivethirtyeightFolderPath + '/fivethirtyeight_ncaa_forecasts (4).csv') #538 file for 2022
both2023_df = pd.read_csv(fivethirtyeightFolderPath + '/fivethirtyeight_ncaa_forecasts (5).csv') #538 file for 2023

In [7]:
#Import labels
mensLabels_df = pd.read_csv(kaggleFolderPath + '/MNCAATourneyDetailedResults.csv')
mensLabels_df = mensLabels_df.sort_values(by=['Season', 'DayNum']).reset_index(drop=True) #ensure ordering of labels to be safe (probably needless line)
womensLabels_df = pd.read_csv(kaggleFolderPath + '/WNCAATourneyDetailedResults.csv')
womensLabels_df = womensLabels_df.sort_values(by=['Season', 'DayNum']).reset_index(drop=True) #ensure ordering of labels to be safe (probably needless line)

#Import team spellings to TeamID map
mensTeamSpellings_df = pd.read_csv(kaggleFolderPath + '/MTeamSpellings.csv', encoding='cp1252')
womensTeamSpellings_df = pd.read_csv(kaggleFolderPath + '/WTeamSpellings.csv', encoding='cp1252')

In [8]:
#Compute head-to-head probabilities for all matches played in 2014 to 2022 for Mens
#and 2015 to 2022 for Womens

def get_round_from_index(i, year, gender):

    #Notice round 1 is the first four round,
    #round 2 is the round of 64 and so on.

    #Notice apart from 2022 and 2023,
    #Womens tournament did not have a first four round.

    if (gender == 'womens' and year < 2022): #no first four round
        return 2 + (i // 32) + (i // 48) + (i // 56) + (i // 60) + (i // 62) #e.g. last game of first round will be index 31, last game of tournament will be index 62
    else: #includes first four round
        if i <= 3: #first four round match
            return 1
        else:
            return 2 +  (i // 36) + (i // 52) + (i // 60) + (i // 64) + (i // 66)

def get_tourneyProb_from_teamID(teamID, givenYear_tourneyProb_df, spellings_df, year):

    givenTeamID_spellings_list = spellings_df.loc[spellings_df['TeamID'] == teamID,'TeamNameSpelling'].tolist()
    givenTeamID_tourneyProb_row = givenYear_tourneyProb_df.loc[[x in givenTeamID_spellings_list for x in givenYear_tourneyProb_df['team_name'].str.lower().tolist()]]

    return givenTeamID_tourneyProb_row

def preprocessed_goto_conversion(listOfOdds, total = 1):
    listOfProbabilities = [1/x for x in listOfOdds]
    isProbSumBelowTotal = sum(listOfProbabilities) < total
    if isProbSumBelowTotal: #input to goto_conversion must be odds where sum of inverse odds exceeds or equals 1
        reversedListOfOdds = [1/(1-x) for x in listOfProbabilities] #unfair odds of losing
        reversedListOfProbabilities = goto_conversion.goto_conversion(reversedListOfOdds, multiplicativeIfImprudentOdds = True) #convert the unfair odds of losing to fair probabilities of losing
        outputListOfProbabilities = [1-x for x in reversedListOfProbabilities] #convert fair probabilities of losing to fair probabilities of winning
    else:
        outputListOfProbabilities = goto_conversion.goto_conversion(listOfOdds, multiplicativeIfImprudentOdds = True)
    return outputListOfProbabilities

def convert_tourneyProb_to_h2hProb(tourneyProb_df, labels_df, spellings_df, year, gender):

    givenYear_labels_df = labels_df.loc[labels_df['Season'] == year]
    correctProbs_list = []

    if year >= 2016: #new format of 538 file from 2016 onwards
        givenYear_tourneyProb_df = tourneyProb_df.loc[tourneyProb_df['gender'] == gender,:]
    else:
        givenYear_tourneyProb_df = tourneyProb_df

    for i in range(givenYear_labels_df.shape[0]): #for each row in tourneyProb_df

        tourneyMatch_row = givenYear_labels_df.iloc[i,:]
        tourneyMatch_round = get_round_from_index(i, year, gender)

        wteam_id = tourneyMatch_row['WTeamID']
        lteam_id = tourneyMatch_row['LTeamID']

        wteam_tourneyProb_row = get_tourneyProb_from_teamID(wteam_id, givenYear_tourneyProb_df, spellings_df, year)
        lteam_tourneyProb_row = get_tourneyProb_from_teamID(lteam_id, givenYear_tourneyProb_df, spellings_df, year)

        isMoreThanOneRow = year >= 2016 #new format of 538 file from 2016 onwards
        if isMoreThanOneRow:

            wteam_firstMatchDay = min(wteam_tourneyProb_row['forecast_date'].tolist())
            lteam_firstMatchDay = min(lteam_tourneyProb_row['forecast_date'].tolist())

            wteam_tourneyProb_row = wteam_tourneyProb_row.loc[wteam_tourneyProb_row['forecast_date'] == wteam_firstMatchDay,:]
            lteam_tourneyProb_row = lteam_tourneyProb_row.loc[lteam_tourneyProb_row['forecast_date'] == lteam_firstMatchDay,:]

            if i == 0:
                print('Key error check to ensure leakage is prevented:')
                display(wteam_tourneyProb_row)
                display(lteam_tourneyProb_row)

        if tourneyMatch_round == 1:
            wteam_winGivenReach_prob = wteam_tourneyProb_row['rd' + str(tourneyMatch_round) + '_win'].values[0]
            lteam_winGivenReach_prob = lteam_tourneyProb_row['rd' + str(tourneyMatch_round) + '_win'].values[0]

        else:
            wteam_winGivenReach_prob = wteam_tourneyProb_row['rd' + str(tourneyMatch_round) + '_win'].values[0] / wteam_tourneyProb_row['rd' + str(tourneyMatch_round - 1) + '_win'].values[0]
            lteam_winGivenReach_prob = lteam_tourneyProb_row['rd' + str(tourneyMatch_round) + '_win'].values[0] / lteam_tourneyProb_row['rd' + str(tourneyMatch_round - 1) + '_win'].values[0]

        convertedProbs = preprocessed_goto_conversion([1/wteam_winGivenReach_prob, 1/lteam_winGivenReach_prob]) #We convert probabilities to odds so we can use goto_conversion
        correctProbs_list.append(convertedProbs[0]) #We use the probability for correct label to calculate score

    return correctProbs_list

#2022
mens2022_correctProbs_list = convert_tourneyProb_to_h2hProb(both2022_df, mensLabels_df, mensTeamSpellings_df, year = 2022, gender = 'mens')
womens2022_correctProbs_list = convert_tourneyProb_to_h2hProb(both2022_df, womensLabels_df, womensTeamSpellings_df, year = 2022, gender = 'womens')

#2021
mens2021_correctProbs_list = convert_tourneyProb_to_h2hProb(both2021_df, mensLabels_df, mensTeamSpellings_df, year = 2021, gender = 'mens')
womens2021_correctProbs_list = convert_tourneyProb_to_h2hProb(both2021_df, womensLabels_df, womensTeamSpellings_df, year = 2021, gender = 'womens')

#2019
mens2019_correctProbs_list = convert_tourneyProb_to_h2hProb(both2019_df, mensLabels_df, mensTeamSpellings_df, year = 2019, gender = 'mens')
womens2019_correctProbs_list = convert_tourneyProb_to_h2hProb(both2019_df, womensLabels_df, womensTeamSpellings_df, year = 2019, gender = 'womens')

print('='*50)

Key error check to ensure leakage is prevented:


,gender,forecast_date,playin_flag,rd1_win,rd2_win,rd3_win,rd4_win,rd5_win,rd6_win,rd7_win,results_to,team_alive,team_id,team_name,team_rating,team_region,team_seed,team_slot
777,mens,2022-03-13,1,0.754715,0.300936,0.087731,0.042307,0.014924,0.004484,0.001805,0,1,84,Indiana,82.98,East,12b,43


,gender,forecast_date,playin_flag,rd1_win,rd2_win,rd3_win,rd4_win,rd5_win,rd6_win,rd7_win,results_to,team_alive,team_id,team_name,team_rating,team_region,team_seed,team_slot
799,mens,2022-03-13,1,0.245285,0.075549,0.017249,0.005429,0.001224,0.000257,0.000077,0,1,2751,Wyoming,78.52,East,12a,42


Key error check to ensure leakage is prevented:


,gender,forecast_date,playin_flag,rd1_win,rd2_win,rd3_win,rd4_win,rd5_win,rd6_win,rd7_win,results_to,team_alive,team_id,team_name,team_rating,team_region,team_seed,team_slot
1608,womens,2022-03-13,1,0.490711,0.182845,0.025318,0.006436,0.000326,0.000063,0.000009,0,1,2168,Dayton,80.91,Greensboro,11a,18


...

,gender,forecast_date,playin_flag,rd1_win,rd2_win,rd3_win,rd4_win,rd5_win,rd6_win,rd7_win,results_to,team_alive,team_id,team_name,team_rating,team_region,team_seed,team_slot
790,mens,2019-03-17,1,0.575381,0.216725,0.088893,0.021065,0.003717,0.000991,0.000279,0,1,2057,Belmont,80.67,East,11a,18


,gender,forecast_date,playin_flag,rd1_win,rd2_win,rd3_win,rd4_win,rd5_win,rd6_win,rd7_win,results_to,team_alive,team_id,team_name,team_rating,team_region,team_seed,team_slot
795,mens,2019-03-17,1,0.424619,0.136819,0.049522,0.012512,0.002353,0.000542,0.000134,0,1,218,Temple,79.39,East,11b,19


Key error check to ensure leakage is prevented:


,gender,forecast_date,playin_flag,rd1_win,rd2_win,rd3_win,rd4_win,rd5_win,rd6_win,rd7_win,results_to,team_alive,team_id,team_name,team_rating,team_region,team_seed,team_slot
1413,womens,2019-03-17,0,1.0,0.688352,0.255351,0.041718,0.008579,0.001428,0.000185,1,1,9,Arizona State,88.31,Portland,5,40


,gender,forecast_date,playin_flag,rd1_win,rd2_win,rd3_win,rd4_win,rd5_win,rd6_win,rd7_win,results_to,team_alive,team_id,team_name,team_rating,team_region,team_seed,team_slot
1432,womens,2019-03-17,0,1.0,0.311648,0.073067,0.00342,0.000248,0.000029,0.000003,1,1,2116,Central Florida,79.89,Portland,12,42


Evidence of Bronze Medal (63rd) with No Risk for 2022 Mens:

In [9]:
#Measure 2022 mens comp score
#Leaderboard: https://www.kaggle.com/competitions/mens-march-mania-2022/leaderboard
#code below is a copy and paste so the logic behind the output score can be viewed without scrolling

logloss = []
for i in range(len(mens2022_correctProbs_list)):
    if i >= 4: #exclude first four matches
        logloss.append(math.log(mens2022_correctProbs_list[i]))
scoreWith01Boost = abs(statistics.mean(logloss) - math.log(0.5)/len(logloss)) #score with 0-1 strategy but no risks
print(scoreWith01Boost)
print('='*50)

0.6064976407396272


Evidence of Bronze Medal (58th) with No Risk for 2022 Womens:

In [10]:
#Measure 2022 womens comp score
#Leaderboard: https://www.kaggle.com/competitions/womens-march-mania-2022/leaderboard
#code below is a copy and paste so the logic behind the output score can be viewed without scrolling

logloss = []
for i in range(len(womens2022_correctProbs_list)):
    if i >= 4: #exclude first four matches
        logloss.append(math.log(womens2022_correctProbs_list[i]))
scoreWith01Boost = abs(statistics.mean(logloss) - math.log(0.5)/len(logloss)) #score with 0-1 strategy but no risks
print(scoreWith01Boost)
print('='*50)

0.42749291920371985


Evidence of Bronze Medal (63rd) with No Risk for 2021 Mens:

In [11]:
#Measure 2021 mens comp score
#Leaderboard: https://www.kaggle.com/competitions/ncaam-march-mania-2021/leaderboard
#code below is a copy and paste so the logic behind the output score can be viewed without scrolling

logloss = []
for i in range(len(mens2021_correctProbs_list)):
    if i >= 4: #exclude first four matches
        logloss.append(math.log(mens2021_correctProbs_list[i]))
scoreWith01Boost = abs(statistics.mean(logloss) - math.log(0.5)/len(logloss)) #score with 0-1 strategy but no risks
print(scoreWith01Boost)
print('='*50)

0.5930164132974177


Evidence of Silver Medal (11th) with No Risk for 2021 Womens:

In [12]:
#Measure 2021 womens comp score
#Leaderboard: https://www.kaggle.com/competitions/ncaaw-march-mania-2021/leaderboard
#code below is a copy and paste so the logic behind the output score can be viewed without scrolling

logloss = []
for i in range(len(womens2021_correctProbs_list)):
    logloss.append(math.log(womens2021_correctProbs_list[i]))
scoreWith01Boost = abs(statistics.mean(logloss) - math.log(0.5)/len(logloss)) #score with 0-1 strategy but no risks
print(scoreWith01Boost)
print('='*50)

0.40810687012546154


Evidence of Top 15% (126th) with No Risk for 2019 Mens:

In [13]:
#Measure 2019 mens comp score
#Leaderboard: https://www.kaggle.com/competitions/mens-machine-learning-competition-2019/leaderboard
#code below is a copy and paste so the logic behind the output score can be viewed without scrolling

logloss = []
for i in range(len(mens2019_correctProbs_list)):
    if i >= 4: #exclude first four matches
        logloss.append(math.log(mens2019_correctProbs_list[i]))
scoreWith01Boost = abs(statistics.mean(logloss) - math.log(0.5)/len(logloss)) #score with 0-1 strategy but no risks
print(scoreWith01Boost)
print('='*50)

0.4645544313328942


Evidence of Silver Medal (22nd) with No Risk for 2019 Womens:

In [14]:
#Measure 2019 womens comp score
#Leaderboard: https://www.kaggle.com/competitions/womens-machine-learning-competition-2019/leaderboard
#code below is a copy and paste so the logic behind the output score can be viewed without scrolling

logloss = []
for i in range(len(womens2019_correctProbs_list)):
    logloss.append(math.log(womens2019_correctProbs_list[i]))
scoreWith01Boost = abs(statistics.mean(logloss) - math.log(0.5)/len(logloss)) #score with 0-1 strategy but no risks
print(scoreWith01Boost)
print('='*50)

0.35257989400721657


# What-If Leaderboard for 2023

In 2023, **I (kaito510) would have won a Gold medal if the risk I took succeeded** and I would have won a Bronze medal if I didn’t take risks. Evidence can be found on the What-If Leaderboard below, the risk I took was SDSU to beat UConn at the Mens Championship Game.

Now, the **purpose of this section is to demonstrate how the risk strategy I took was near-optimal**, using both numerical and then theoretical evidence. I understand there are possibly other past gold medalists that can also say that they would’ve won another gold medal if one key match went their way.

PS: Credit to Tim Yee for the leaderboard snapshots

In [15]:
#Get Inputs
after125games_df = pd.read_csv('/kaggle/input/mwncaa-lb-raw/Leaderboard_History/march-machine-learning-mania-2023-publicleaderboard_62_63_1157_2023-04-03.csv')
after125games_df['SumScore'] = after125games_df['Score'] * 125
#print('='*50)
#print('display(after125games_df)')
#display(after125games_df)
after126games_df = pd.read_csv('/kaggle/input/mwncaa-lb-raw/Leaderboard_History/march-machine-learning-mania-2023-publicleaderboard_63_63_0924_2023-04-04.csv')
after126games_df['SumScore'] = after126games_df['Score'] * 126
#print('='*50)
#print('display(after126games_df)')
#display(after126games_df)

In [16]:
#Join Leaderboards
leaderboards_df = pd.merge(after125games_df, after126games_df, on='TeamId', suffixes=('_after125', '_after126'))
print('='*50)
print('TOP 10 OF 2023 ORIGINAL LEADERBOARD')
display(leaderboards_df[['TeamName_after126', 'Score_after126']].head(10))

#Compute Leaderoard if Mens Championship Game was flipped
leaderboards_df['lastGameScore'] = leaderboards_df['SumScore_after126'] - leaderboards_df['SumScore_after125']
leaderboards_df['flippedLastGameScore'] = (1 - (leaderboards_df['lastGameScore'] ** 0.5)) ** 2
leaderboards_df['Score_lastGameFlipped'] = (leaderboards_df['SumScore_after125'] + leaderboards_df['flippedLastGameScore'])/126
leaderboards_df = leaderboards_df.sort_values(by='Score_lastGameFlipped')
leaderboards_df = leaderboards_df.reset_index(drop=True)
print('='*50)
print('TOP 10 OF 2023 LEADERBOARD IF MENS CHAMPIONSHIP GAME WAS FLIPPED')
print('NOTICE kaito510 RANKS IN GOLD MEDAL ZONE (INSIDE TOP 10)')
print('THE FULL WHAT-IF LEADERBOARD IS AVAILABLE AS AN OUTPUT CSV FILE')
leaderboards_df = leaderboards_df[['TeamName_after126', 'Score_after126', 'Score_lastGameFlipped']]
display(leaderboards_df.iloc[:10])
leaderboards_df.to_csv('WhatIfLeaderboard2023.csv', index_label='What-If Leaderboard Rank')

TOP 10 OF 2023 ORIGINAL LEADERBOARD


,TeamName_after126,Score_after126
0,RustyB,0.17371
1,NicholasHilton,0.17557
2,tihonby,0.17525
3,mbund1237,0.17619
4,Matthias,0.17522
5,David Scott,0.17651
6,Jack Lichtenstein,0.17704
7,cjwh,0.17800
8,FNiemann,0.17854
9,maze508,0.17843


TOP 10 OF 2023 LEADERBOARD IF MENS CHAMPIONSHIP GAME WAS FLIPPED
NOTICE kaito510 RANKS IN GOLD MEDAL ZONE (INSIDE TOP 10)
THE FULL WHAT-IF LEADERBOARD IS AVAILABLE AS AN OUTPUT CSV FILE


,TeamName_after126,Score_after126,Score_lastGameFlipped
0,RustyB,0.17371,0.174890
1,NicholasHilton,0.17557,0.176031
2,mbund1237,0.17619,0.176231
3,tihonby,0.17525,0.176447
4,David Scott,0.17651,0.178043
5,Matthias,0.17522,0.178069
6,FNiemann,0.17854,0.179293
7,cjwh,0.17800,0.179602
8,Jack Lichtenstein,0.17704,0.180430
9,kaito510,0.18837,0.180437


The What-If Leaderboard assumes that the same submission (out of a maximum of two for each participant) had the better score after 125/126 games and after 126/126 games for all participants.

**In 2023, we were evaluated under the Brier Score. Below is a mathematical proof that the optimal risk strategy to win a medal under Brier Score is when we assume a team with 33.3% chance of winning a match to win that match.**

The expected return when we risk on a given game can be expressed as:

f(p) = p(1 - p)^2 where p is the probability of success and (1-p)^2 is essentially the reward for the risk taken if the risk succeeds

This implies f'(p) and f''(p) can be expressed as:

f'(p) = -2p + 2p^2 + (1-p)^2

f''(p) = -4 + 6p

argmax_p f(p) = 1/3 with tedious mathematical working omitted.

Thus, expected reward is maximised when we assume a team with 1/3 chance of winning a match to win that match. The risk I took was near this optimal.

**Here is the same proof for Log-Loss:**

The expected return when we risk on a given game can be expressed as:

f(p) = -p*ln(p) where p is the probability of success and -ln(p) is essentially the reward for the risk taken if the risk succeeds

This implies f'(p) and f''(p) can be expressed as:

f'(p) = -ln(p)-1

f''(p) = -1/p

argmax_p f(p) = 1/e ~= 36.8% with tedious mathematical working omitted.

Thus, expected reward is maximised when we assume a team with a 36.8% chance of winning a match to win that match.